In [ ]:
'''Mass Email Script for WFU MSBA Alumni Outreach
Reich x Claude 05.21.2026
------------------------------------------------
Prerequisites:
    pip install pandas openpyxl
 
Gmail setup (recommended):
    1. Enable 2-Factor Authentication on your Google account
    2. Go to: myaccount.google.com > Security > App Passwords
    3. Create an App Password for "Mail" — use that as your GMAIL_APP_PASSWORD below
    4. Never use your real Gmail password here
 
Excel file must have these columns:
    First Name, Last Name, Industry, Job Function, Employer, Job Title, Email
 
Run:
    python Alumni_Email_Mass_Send.ipynb
'''

### Imports and Configuration

In [2]:
import pandas as pd
import smtplib
import time
import logging
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from pathlib import Path
 
# ─────────────────────────────────────────────
# CONFIGURATION — edit these before running
# ─────────────────────────────────────────────
 
EXCEL_FILE       = "file.xlsx"                                           # Path to your Excel file
SHEET_NAME       = 0                                                     # Sheet index (0 = first) or sheet name string
 
GMAIL_ADDRESS      = "your.mail@gmail.com"                               # Personal Gmail you send FROM
GMAIL_APP_PASSWORD = "xxxx xxxx xxxx xxxx"                               # 16-char Gmail App Password
REPLY_TO_ADDRESS   = "your.mail@gmail.com"                               # Personal email (alumni or personal) — replies will land here
 
SENDER_NAME      = "Your Name"
DELAY_SECONDS    = 2                                                     # Pause between emails — avoids spam filters (keep >= 2)
DRY_RUN          = True                                                  # True = print emails without sending; False = actually send
 
LOG_FILE         = "email_log.csv"                                       # Tracks sent/failed emails for resuming
 
# Column name mapping — update if your Excel headers differ
COL_FIRST_NAME   = "First Name"
COL_LAST_NAME    = "Last Name"
COL_INDUSTRY     = "Industry"
COL_JOB_FUNCTION = "Job Function"
COL_EMPLOYER     = "Employer"
COL_JOB_TITLE    = "Job Title"
COL_EMAIL        = "Email"

### Function Definitions

In [3]:
# ─────────────────────────────────────────────
# EMAIL TEMPLATE
# ─────────────────────────────────────────────
 
def build_email(row: pd.Series) -> tuple[str, str]:
    """Returns (subject, body) for a single contact."""
    subject = "WFU MSBA Alum – Quick Question About Your Career Path"

    job_title = str(row[COL_JOB_TITLE].strip())
    if 'Intern' in job_title: job_title_phrase = ""
    else: 
        job_title_phrase = (
            f"as an {job_title} " if starts_with_vowel(job_title) else f"as a {job_title} "
        )
    
    industry = str(row[COL_INDUSTRY]).strip()
    industry_phrase = (
        f" in the {industry} industry" if industry and industry.lower() not in ("other", "nan", "") 
        else ""
    )
    
    job_fcn = str(row[COL_JOB_FUNCTION]).strip()
    job_fcn_phrase = ""
    if not job_fcn or job_fcn.lower() in ("other", "nan") or 'Intern' in job_fcn: 
        job_fcn_phrase = f"in your field"                                 # unknown field 
    elif job_fcn.startswith("Analyst"): 
        job_fcn_phrase = f"as an {job_fcn}"                               # "as an Analyst"
    elif any(word in job_fcn for word in ("Analyst", "Scientist")): 
        job_fcn_phrase = f"as a {job_fcn}"                                # "as a Data Analyst"
    else: job_fcn_phrase = f"in {job_fcn.lower()}"                        # "in Analytics"
    
    body = f"""Dear {row[COL_FIRST_NAME]} {row[COL_LAST_NAME]},

I just graduated from MSBA '26, and I see you also graduated from WFU MSBA!

Would you be willing to speak with me for 15 minutes about your experience{industry_phrase} working {job_title_phrase}at {row[COL_EMPLOYER]}? I am interested in pursuing a career {job_fcn_phrase} and would love to hear from you.

Thank you for your time, and I look forward to getting in touch!

Regards,
{SENDER_NAME}"""
 
    return subject, body
 
 
# ─────────────────────────────────────────────
# LOGGING SETUP
# ─────────────────────────────────────────────
 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)s  %(message)s",
    handlers=[
        logging.FileHandler("email_run.log"),
        logging.StreamHandler(),
    ],
)
log = logging.getLogger(__name__)
 
 
# ─────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────

def starts_with_vowel(job_title):
    return job_title[0].lower() in {'a', 'e', 'i', 'o', 'u'}

def load_contacts(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=SHEET_NAME, dtype=str)
    df.columns = df.columns.str.strip()
    df = df.dropna(subset=[COL_EMAIL])
    df[COL_EMAIL] = df[COL_EMAIL].str.strip()
    df = df[df[COL_EMAIL] != ""]
    log.info(f"Loaded {len(df)} contacts with emails from '{path}'")
    return df
 
 
def load_sent_emails(log_path: str) -> set:
    """Load previously sent emails so we can skip them on re-runs."""
    p = Path(log_path)
    if not p.exists():
        return set()
    sent_df = pd.read_csv(p, dtype=str)
    already_sent = set(sent_df[sent_df["status"] == "sent"]["email"].str.lower())
    log.info(f"Skipping {len(already_sent)} already-sent addresses from previous run")
    return already_sent
 
 
def log_result(log_path: str, email: str, name: str, status: str, note: str = ""):
    row = pd.DataFrame([{"email": email, "name": name, "status": status, "note": note}])
    write_header = not Path(log_path).exists()
    row.to_csv(log_path, mode="a", index=False, header=write_header)
 
 
def send_email(smtp: smtplib.SMTP_SSL, to_address: str, subject: str, body: str):
    msg = MIMEMultipart("alternative")
    msg["Subject"]  = subject
    msg["From"]     = f"{SENDER_NAME} <{GMAIL_ADDRESS}>"
    msg["To"]       = to_address
    msg["Reply-To"] = f"{SENDER_NAME} <{REPLY_TO_ADDRESS}>"
    msg.attach(MIMEText(body, "plain"))
    smtp.sendmail(GMAIL_ADDRESS, to_address, msg.as_string())
 
 
# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
 
def main():
    df = load_contacts(EXCEL_FILE)
    already_sent = load_sent_emails(LOG_FILE)
 
    to_send = df[~df[COL_EMAIL].str.lower().isin(already_sent)]
    log.info(f"{len(to_send)} emails to send this run")
 
    if DRY_RUN:
        log.info("DRY RUN MODE — no emails will actually be sent\n")
 
    sent_count = 0
    fail_count = 0
 
    ctx = smtplib.SMTP_SSL("smtp.gmail.com", 465) if not DRY_RUN else None
    if ctx:
        ctx.login(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
        log.info("Logged in to Gmail SMTP")
 
    try:
        for _, row in to_send.iterrows():
            #if sent_count > 0:
            #    break
            name  = f"{row[COL_FIRST_NAME]} {row[COL_LAST_NAME]}"
            email = row[COL_EMAIL]
 
            try:
                subject, body = build_email(row)
 
                if DRY_RUN:
                    print(f"\n{'─'*60}")
                    print(f"TO:      {name} <{email}>")
                    print(f"SUBJECT: {subject}")
                    print(body)
                else:
                    send_email(ctx, email, subject, body)
                    time.sleep(DELAY_SECONDS)
 
                log_result(LOG_FILE, email, name, "sent")
                sent_count += 1
 
            except Exception as e:
                log.error(f"  FAILED for {name} <{email}>: {e}")
                log_result(LOG_FILE, email, name, "failed", str(e))
                fail_count += 1
 
    finally:
        if ctx:
            ctx.quit()
 
    log.info(f"\n{'='*40}")
    log.info(f"Done.  Sent: {sent_count}  |  Failed: {fail_count}")
    log.info(f"Full log saved to: {LOG_FILE}")

2026-05-20 15:09:22,280  INFO  Loaded 132 contacts with emails from 'Where are they now WFU alumni May 2026.xlsx'
2026-05-20 15:09:22,281  INFO  132 emails to send this run
2026-05-20 15:09:22,470  INFO  Logged in to Gmail SMTP
2026-05-20 15:15:04,940  INFO  
2026-05-20 15:15:04,941  INFO  Done.  Sent: 132  |  Failed: 0
2026-05-20 15:15:04,942  INFO  Full log saved to: email_log.csv


### Test Cases

In [ ]:
# ── Test cases ──────────────────────────────────────────
test_cases = [
    {"First Name": "John", "Last Name": "Doe",  COL_INDUSTRY: "Consulting",  COL_JOB_FUNCTION: "Consulting",     COL_JOB_TITLE: "Staff Consultant", COL_EMPLOYER: "Deloitte"},
    {"First Name": "Jane", "Last Name": "Smith", COL_INDUSTRY: "Other",       COL_JOB_FUNCTION: "Data Science",   COL_JOB_TITLE: "Analyst",          COL_EMPLOYER: "Google"},
    {"First Name": "Bob",  "Last Name": "Lee",   COL_INDUSTRY: "",            COL_JOB_FUNCTION: "Other",          COL_JOB_TITLE: "Engineer",         COL_EMPLOYER: "Apple"},
    {"First Name": "Sara", "Last Name": "Kim",   COL_INDUSTRY: "Finance",     COL_JOB_FUNCTION: "Data Analyst",   COL_JOB_TITLE: "Associate",        COL_EMPLOYER: "JPMorgan"},
    {"First Name": "Alex", "Last Name": "Park",  COL_INDUSTRY: "Analytics",   COL_JOB_FUNCTION: "Intern",         COL_JOB_TITLE: "Analytics Intern", COL_EMPLOYER: "IBM"},
]

for t in test_cases:
    row = pd.Series(t)
    subject, body = build_email(row)
    print(body)
    print("─" * 60)

In [ ]:
# Preview raw output with visible newlines
subject, body = build_email(pd.Series(test_cases[0]))
print(repr(body))

In [ ]:
print(f"DRY_RUN: {DRY_RUN}")
print(f"Sending from: {GMAIL_ADDRESS}")
print(f"Reply-To: {REPLY_TO_ADDRESS}")
print(f"App password set: {GMAIL_APP_PASSWORD != 'xxxx xxxx xxxx xxxx'}")

### Run

In [ ]:
if __name__ == "__main__":
    main()